# 🧪 ChaiFolder Tutorial — A Student's Guide to Protein Structure Prediction & Design

## What you'll learn

This notebook teaches you how to use **ChaiFolder** (a wrapper around [Chai-1](https://chailab.bio/)) for:

1. **Basic structure prediction** — fold a protein from sequence
2. **Prediction with templates** — use a known structure to guide prediction
3. **Prediction with ESM embeddings** — use protein language model features
4. **Inverse folding with LigandMPNN** — design a sequence for a structure
5. **Iterative design loop** — combine prediction + design in an optimization cycle

### Key Concepts

| Concept | What it does |
|---|---|
| **ChaiFolder** | Wrapper that breaks Chai-1 into modular steps: `prep_inputs → get_embeddings → run_trunk → sample → save` |
| **Templates** | Feed a known 3D structure to guide prediction (like giving the model a hint) |
| **ESM embeddings** | Protein language model features that encode evolutionary information |
| **Partial diffusion** | Start denoising from a noisy version of a previous prediction (refinement) |
| **LigandMPNN** | Given a 3D backbone, design a sequence that would fold into it |
| **Hysteresis** | Carrying information from one prediction to the next in an iterative loop |

---

**Runtime**: This notebook requires a **GPU** (T4 or better). In Colab: `Runtime → Change runtime type → GPU`.

## 1. Installation

This cell installs Chai-1, downloads model weights (~5GB), and sets up LigandMPNN. Takes ~5-10 minutes on first run.

In [ ]:
# @title 1. Install Dependencies (run once)
import importlib.util

if importlib.util.find_spec("chai_lab") is not None:
  print("Chai is already installed!")
else:
  import os
  import subprocess
  import threading

  def setup_models():
      os.system("apt-get install -y -qq aria2")
      base_url = "http://files.ipd.uw.edu/pub/protein_hunter/chai1"
      dl_dir = "/usr/local/lib/python3.12/dist-packages/downloads"
      os.makedirs(f"{dl_dir}/models_v2", exist_ok=True)
      os.makedirs(f"{dl_dir}/esm", exist_ok=True)

      downloads = [
          (f"{dl_dir}", "conformers_v1.apkl", "-x16 -s16"),
          (f"{dl_dir}/esm", "esm2/traced_sdpa_esm2_t36_3B_UR50D_fp16.pt", "-x16 -s16"),
          (f"{dl_dir}/models_v2", "models_v2/trunk.pt", "-x16 -s16"),
          (f"{dl_dir}/models_v2", "models_v2/diffusion_module.pt", "-x16 -s16"),
          (f"{dl_dir}/models_v2", "models_v2/confidence_head.pt", "-x8"),
          (f"{dl_dir}/models_v2", "models_v2/feature_embedding.pt", ""),
          (f"{dl_dir}/models_v2", "models_v2/token_embedder.pt", ""),
          (f"{dl_dir}/models_v2", "models_v2/bond_loss_input_proj.pt", "")
      ]
      for target_dir, url_path, opts in downloads:
          url = f"{base_url}/{url_path}"
          subprocess.run(f"aria2c {opts} --dir={target_dir} {url}", shell=True, check=True)

  print("Starting model downloads in background...")
  download_thread = threading.Thread(target=setup_models)
  download_thread.start()

  print("Installing chai-lab...")
  os.system("pip install --no-deps git+https://github.com/sokrypton/chai-lab.git "
    "'gemmi~=0.6.3' 'jaxtyping>=0.2.25' 'pandera>=0.24' 'antipickle==0.2.0' "
    "'rdkit~=2024.9.5' 'modelcif>=1.0' 'biopython>=1.83' typing_inspect "
    "ihm mypy_extensions equinox wadler_lindig py3Dmol")

  print("Installing LigandMPNN...")
  os.system("git clone https://github.com/sokrypton/LigandMPNN.git")
  os.system("mkdir -p model_params")
  os.system("bash LigandMPNN/get_model_params.sh model_params")
  os.system("pip install git+https://github.com/prody/ProDy.git")
  os.system("pip install ml_collections")
  os.system("pip install git+https://github.com/sokrypton/py2Dmol.git")

  print("Waiting for model downloads to finish...")
  download_thread.join()
  print("✅ Setup complete!")

Starting model downloads in background...
Installing chai-lab...
Installing LigandMPNN...
Waiting for model downloads to finish...
✅ Setup complete!


## 2. Load Libraries & Initialize ChaiFolder

All ChaiFolder code is bundled inside the `chai_lab` package installed above.
One import is all you need.

In [ ]:
# @title 2. Import ChaiFolder

import gc
import torch

from chai_lab.folding import (
    ChaiFolder,
    LigandMPNNWrapper,
    optimize_protein_design,
    sample_seq,
    clean_protein_sequence,
    compute_ca_rmsd,
    get_backbone_coords_from_result,
    prepare_refinement_coords,
)

## 3. Initialize the Models

This loads all Chai-1 neural network weights onto the GPU (~30s).

In [ ]:
# @title 3. Initialize ChaiFolder & LigandMPNN
folder = ChaiFolder(device="cuda:0")
designer = LigandMPNNWrapper()
print("✅ Models loaded and ready!")

✅ Models loaded and ready!


---

## Example 1: Basic Structure Prediction (No Templates, No ESM)

The simplest use case — give ChaiFolder a protein sequence and predict its 3D structure.

### How `prep_inputs` works

Each chain is specified as a list: `[sequence, chain_name, entity_type, options_dict]`

```python
sequences = [
    ["MKTLLILAS...",  # amino acid sequence
     "A",             # chain name
     "protein",       # entity type: "protein", "ligand", "rna", "dna"
     {}]              # options dict (empty = defaults)
]
```

### The prediction pipeline

```
prep_inputs() → get_embeddings() → run_trunk() → sample() → save()
```

In [ ]:
# @title Example 1: Predict structure of a small protein
# A small WW domain sequence (~35 residues) — fast to predict
sequence = "GSKLPPGWEKRMSRSSGRVYYFNHITNASQWERP"

print(f"Sequence: {sequence}")
print(f"Length:   {len(sequence)} residues")
print()

# Define the input: [sequence, chain_name, entity_type, options]
# Empty options dict = no ESM, no templates, no cyclic
sequences = [
    [sequence, "A", "protein", {}]
]

# Step-by-step prediction
print("Step 1: Preparing inputs (tokenize, build features)...")
folder.prep_inputs(sequences)

print("Step 2: Computing embeddings...")
folder.get_embeddings()

print("Step 3: Running trunk (3 recycles)...")
folder.run_trunk(num_trunk_recycles=3)

print("Step 4: Sampling structure (200 diffusion steps)...")
folder.sample(num_diffn_timesteps=200)

# Save and inspect results
folder.save("example1_basic.cif")
result = folder.state.result

print()
print("=== Results ===")
print(f"  pLDDT (confidence):  {result['plddt'].mean().item()*100:.1f}")
print(f"  pTM:                 {result['ptm'].item():.3f}")
print(f"  Ranking score:       {result['ranking_score']:.3f}")
print()
print("Higher pLDDT (>70) and pTM (>0.5) = more confident prediction")

Sequence: GSKLPPGWEKRMSRSSGRVYYFNHITNASQWERP
Length:   34 residues

Step 1: Preparing inputs (tokenize, build features)...
Step 2: Computing embeddings...
Step 3: Running trunk (3 recycles)...
Step 4: Sampling structure (200 diffusion steps)...

=== Results ===
  pLDDT (confidence):  82.2
  pTM:                 0.624
  Ranking score:       0.125

Higher pLDDT (>70) and pTM (>0.5) = more confident prediction


In [ ]:
# @title Visualize Example 1
folder.plot(color_by="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Example 1b: Distogram Output

The distogram is a **predicted inter-residue distance distribution** computed from
the trunk pair representation — available immediately after `run_trunk()`, before
diffusion sampling.

For each pair of residues (i, j) the distogram gives a probability distribution
over 64 distance bins. Lower bins correspond to shorter distances (residues close
in 3D space); higher bins to longer distances.

| What | Code | Shape |
|---|---|---|
| Probabilities (softmax) | `folder.get_distogram()` | `[n_tokens, n_tokens, 64]` |
| Raw logits | `folder.get_distogram(as_probs=False)` | `[n_tokens, n_tokens, 64]` |
| Most-likely bin | `.argmax(-1)` | `[n_tokens, n_tokens]` |
| Contact probability | `.sum over first k bins` | `[n_tokens, n_tokens]` |

The distogram is **independent of diffusion** — it reflects the trunk's geometric
understanding of the sequence, not the sampled structure.

In [ ]:
# @title Example 1b: Access and visualize the distogram
import matplotlib.pyplot as plt

# Distogram is computed during run_trunk() — no extra inference needed.
# Returns softmax probabilities: [n_tokens, n_tokens, n_dist_bins]
disto = folder.get_distogram()  # as_probs=True by default
print(f"Distogram shape: {list(disto.shape)}")
print(f"  {disto.shape[0]} residues x {disto.shape[1]} residues x {disto.shape[2]} distance bins")
print(f"  Probabilities sum to 1.0: {disto.sum(-1).mean().item():.4f}")

# Most-likely bin index for each residue pair — a proxy for relative distance.
# Higher index = predicted to be farther apart.
pred_bin = disto.argmax(-1).float().numpy()   # [n_tokens, n_tokens]

# Contact probability: P(residue pair is in one of the closer bins).
# We use the first quarter of bins as a "close contact" heuristic.
n_contact_bins = disto.shape[-1] // 4          # bins 0-15 out of 0-63
contact_prob = disto[..., :n_contact_bins].sum(-1).numpy()  # [n_tokens, n_tokens]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(pred_bin, cmap="RdYlBu_r", origin="upper")
axes[0].set_title("Most-likely distance bin\n(low = close, high = far)")
axes[0].set_xlabel("Residue index")
axes[0].set_ylabel("Residue index")
plt.colorbar(im0, ax=axes[0], label="Bin index (0 = closest)")

im1 = axes[1].imshow(contact_prob, cmap="Reds", vmin=0, vmax=1, origin="upper")
axes[1].set_title(f"Predicted contact probability\n(P of being in bins 0-{n_contact_bins-1})")
axes[1].set_xlabel("Residue index")
axes[1].set_ylabel("Residue index")
plt.colorbar(im1, ax=axes[1], label="Probability")

plt.suptitle(f"Distogram — {disto.shape[0]}-residue protein", y=1.02)
plt.tight_layout()
plt.show()

print()
print("=== Distogram statistics ===")
print(f"  Modal bin (mean over all pairs):      {pred_bin.mean():.1f} / {disto.shape[-1]-1}")
print(f"  Fraction of pairs in contact bins:    {contact_prob.mean():.3f}")
print(f"  Sequence-adjacent contacts (|i-j|=1): {contact_prob.diagonal(1).mean():.3f}  (expect ~1.0)")

gc.collect(); torch.cuda.empty_cache()

---

## Example 2: Prediction WITH Templates

Templates are known 3D structures that guide the prediction — like giving the model a "hint" about the expected fold.

### When to use templates
- You have a homologous structure (e.g., from PDB)
- You're refining a previous prediction
- You're doing iterative design (each round uses the previous structure as template)

### Template options
```python
options = {
    "template_pdb": "path/to/template.cif",     # path to template structure
    "template_chain_id": "A",                     # which chain in the template
    "randomize_template_sequence": True,          # hide sequence identity (for design)
}
```

### Why randomize template sequence?
In design, you want the model to use the **geometry** from the template but not be biased by the template's **sequence**. Setting `randomize_template_sequence=True` replaces the template residue types with random ones.

In [ ]:
# @title Example 2: Predict with template guidance
sequence = "GSKLPPGWEKRMSRSSGRVYYFNHITNASQWERP"

# --- Prediction WITHOUT template (baseline) ---
print("=== Without template ===")
folder.prep_inputs([[sequence, "A", "protein", {}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("example2_no_template.cif")
score_no_tpl = folder.state.result["ranking_score"]
plddt_no_tpl = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {plddt_no_tpl:.1f}  |  Score: {score_no_tpl:.3f}")

gc.collect(); torch.cuda.empty_cache()

# --- Prediction WITH template (using previous result as guide) ---
print()
print("=== With template ===")
template_opts = {
    "template_pdb": "example2_no_template.cif",
    "template_chain_id": "A",
    "randomize_template_sequence": False,  # keep original sequence
}

folder.prep_inputs([[sequence, "A", "protein", template_opts]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("example2_with_template.cif")
score_with_tpl = folder.state.result["ranking_score"]
plddt_with_tpl = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {plddt_with_tpl:.1f}  |  Score: {score_with_tpl:.3f}")

print()
print("Templates often improve confidence, especially for iterative refinement.")

=== Without template ===
  pLDDT: 81.5  |  Score: 0.123

=== With template ===
  pLDDT: 92.3  |  Score: 0.153

Templates often improve confidence, especially for iterative refinement.


In [ ]:
# @title Visualize Example 2 (with template)
folder.plot(color_by="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Example 3: Prediction WITH ESM Embeddings

ESM2 is a protein language model trained on millions of sequences. It provides **evolutionary context** — information about what amino acids tend to appear at each position.

### When to use ESM
- For single-sequence predictions (no MSA available)
- For designed sequences that have no natural homologs
- ESM is especially useful for ligand binder design

### ESM options
```python
options = {
    "use_esm": True,                # compute ESM embeddings for this chain
    "replace_x_with_mask": True,    # treat X residues as <mask> tokens
}
```

In [ ]:
# @title Example 3: Predict with ESM embeddings
sequence = "GSKLPPGWEKRMSRSSGRVYYFNHITNASQWERP"

# --- Without ESM (baseline) ---
print("=== Without ESM ===")
folder.prep_inputs([[sequence, "A", "protein", {}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
score_no_esm = folder.state.result["ranking_score"]
plddt_no_esm = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {plddt_no_esm:.1f}  |  Score: {score_no_esm:.3f}")

gc.collect(); torch.cuda.empty_cache()

# --- With ESM ---
print()
print("=== With ESM embeddings ===")
folder.prep_inputs([[sequence, "A", "protein", {"use_esm": True}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("example3_with_esm.cif")
score_esm = folder.state.result["ranking_score"]
plddt_esm = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {plddt_esm:.1f}  |  Score: {score_esm:.3f}")

print()
print("ESM provides evolutionary context from the protein language model.")

=== Without ESM ===
  pLDDT: 82.1  |  Score: 0.125

=== With ESM embeddings ===
  pLDDT: 95.5  |  Score: 0.160

ESM provides evolutionary context from the protein language model.


---

## Example 4: Inverse Folding with LigandMPNN

Given a 3D structure, **LigandMPNN** designs a sequence that should fold into it.

```
Structure Prediction:  sequence → structure  (ChaiFolder)
Inverse Folding:       structure → sequence  (LigandMPNN)
```

### Key parameters
- **`temperature`**: Low (0.01) = conservative, High (1.0) = diverse
- **`chains_to_design`**: Which chains to redesign (e.g., "A" or "B")
- **`model_type`**: `"soluble_mpnn"` (proteins) or `"ligand_mpnn"` (with ligands)

In [ ]:
# @title Example 4: Design a sequence for a predicted structure

# First, predict a structure from a random sequence
print("Step 1: Fold a random 80-residue sequence...")
random_seq = sample_seq(80, frac_X=0.0)
print(f"  Random: {random_seq[:50]}...")

folder.prep_inputs([[random_seq, "A", "protein", {}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("example4_random_fold.cif")

initial_score = folder.state.result["ranking_score"]
initial_plddt = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {initial_plddt:.1f}  |  Score: {initial_score:.3f}")

gc.collect(); torch.cuda.empty_cache()

# Use LigandMPNN to design a better sequence for this backbone
print()
print("Step 2: Design sequence with LigandMPNN...")
designed_sequences = designer.run(
    pdb_path="example4_random_fold.cif",
    model_type="soluble_mpnn",
    temperature=0.1,
    chains_to_design="A",
    seed=42,
    extra_args={
        "--batch_size": 1,
        "--checkpoint_soluble_mpnn": "./model_params/solublempnn_v_48_020.pt",
    }
)
designed_seq = designed_sequences[0]
print(f"  Designed: {designed_seq[:50]}...")

# Fold the designed sequence
print()
print("Step 3: Fold the designed sequence...")
folder.prep_inputs([[designed_seq, "A", "protein", {}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("example4_designed_fold.cif")

designed_score = folder.state.result["ranking_score"]
designed_plddt = folder.state.result["plddt"].mean().item() * 100
print(f"  pLDDT: {designed_plddt:.1f}  |  Score: {designed_score:.3f}")

print()
delta = designed_score - initial_score
print(f"Score change: {delta:+.3f} {'✅ improved!' if delta > 0 else '(decreased)'}")

Step 1: Fold a random 80-residue sequence...
  Random: GMKYMKRESQRCRGQGGVNTKIRKMGFRAGNFVDSRERNGTWSIRCHGDC...
  pLDDT: 44.3  |  Score: 0.053

Step 2: Design sequence with LigandMPNN...
  Designed: MKQWVKKGTDGATGPQATDPLLRSLGFGGGTITAAKEVDGIYYIHVDGTS...

Step 3: Fold the designed sequence...
  pLDDT: 60.8  |  Score: 0.104

Score change: +0.051 ✅ improved!


In [ ]:
# @title Visualize designed structure
folder.plot(color_by="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Example 5: Iterative Design Loop (The Full Protocol)

This is where it all comes together! The iterative loop alternates between:
1. **Fold** the current sequence (with optional templates/ESM from previous round)
2. **Design** a new sequence using LigandMPNN on the predicted structure
3. **Repeat**

```
Random Seq → [Fold] → Structure → [MPNN] → Better Seq → [Fold+Template] → ...
                                      ↑                                      │
                                      └──────────────────────────────────────┘
```

### Hysteresis Modes (How info flows between iterations)

| Mode | How it works | Best for |
|---|---|---|
| `templates` | Previous structure → template input, weighted by PDE confidence | Unconditional, protein binders |
| `esm` | ESM2 embeddings provide evolutionary context each round | Ligand binders |
| `partial_diffusion` | Start diffusion from noisy previous coords (refinement) | Refinement |
| `none` | No information carried forward | Baseline comparison |

In [ ]:
# @title Example 5a: Simple manual design loop (educational version)
# This shows the loop step-by-step so you understand each piece.

# @markdown ### Settings
length = 80           # @param {type:"integer"}
n_cycles = 3          # @param {type:"integer"}
hysteresis = "templates"  # @param ["templates", "esm", "partial_diffusion", "none"]
temperature = 0.1     # @param {type:"number"}

# Generate random starting sequence
seq = sample_seq(length, frac_X=0.0)
print(f"Starting sequence: {seq[:50]}...")
print(f"Hysteresis mode:   {hysteresis}")
print()

# ==== Step 0: Initial fold (no templates, no history) ====
print("=" * 60)
print("Step 0: Initial fold (no templates)")
print("=" * 60)

folder.prep_inputs([[seq, "A", "protein", {"use_esm": hysteresis == "esm"}]])
folder.get_embeddings()
folder.run_trunk(num_trunk_recycles=3)
folder.sample(num_diffn_timesteps=200)
folder.save("loop_step0.cif")

result = folder.state.result
print(f"  pLDDT: {result['plddt'].mean().item()*100:.1f}")
print(f"  Score: {result['ranking_score']:.3f}")

prev_state = folder.save_state()
prev_pdb = "loop_step0.cif"
best_score = result["ranking_score"]
best_seq = seq
best_step = 0

gc.collect(); torch.cuda.empty_cache()

# ==== Optimization cycles ====
for step in range(1, n_cycles + 1):
    print()
    print("=" * 60)
    print(f"Step {step}: Design + Fold")
    print("=" * 60)

    # 1) DESIGN: LigandMPNN designs a new sequence for the previous structure
    print("  [MPNN] Designing new sequence...")
    new_seqs = designer.run(
        pdb_path=prev_pdb,
        model_type="soluble_mpnn",
        temperature=temperature,
        chains_to_design="A",
        seed=42 + step,
        extra_args={
            "--batch_size": 1,
            "--checkpoint_soluble_mpnn": "./model_params/solublempnn_v_48_020.pt",
        }
    )
    seq = new_seqs[0]
    print(f"  New seq: {seq[:50]}...")

    # 2) FOLD with hysteresis — carry info from previous round
    opts = {}
    template_weight = None

    if hysteresis == "templates":
        # Use previous structure as template, weighted by confidence
        opts = {
            "template_pdb": prev_pdb,
            "template_chain_id": "A",
            "randomize_template_sequence": True,
        }
        # PDE-based template weighting: confident regions get stronger templates
        pde = prev_state.result["pde"]
        pde_bins = (_bin_centers(0.0, 32.0, 64) <= 1.5).sum().item()
        template_weight = pde[..., :pde_bins].sum(-1)

    elif hysteresis == "esm":
        opts = {"use_esm": True}

    elif hysteresis == "partial_diffusion":
        opts = {}  # partial diffusion handled below

    print(f"  [Fold] Predicting structure (hysteresis={hysteresis})...")
    folder.prep_inputs([[seq, "A", "protein", opts]])
    folder.get_embeddings()
    folder.run_trunk(num_trunk_recycles=3, template_weight=template_weight)

    # Handle partial diffusion: start from noisy previous coords
    if hysteresis == "partial_diffusion" and prev_state.result is not None:
        refine_coords = prepare_refinement_coords(
            folder, prev_state.result, prev_state.batch_inputs
        )
        folder.sample(
            num_diffn_timesteps=200,
            refine_from_coords=refine_coords,
            refine_from_step=100,  # start from 50% noise
        )
    else:
        folder.sample(num_diffn_timesteps=200)

    prev_pdb = f"loop_step{step}.cif"
    folder.save(prev_pdb)
    prev_state = folder.save_state()

    result = folder.state.result
    score = result["ranking_score"]
    plddt = result["plddt"].mean().item() * 100
    print(f"  pLDDT: {plddt:.1f}  |  Score: {score:.3f}")

    if score > best_score:
        best_score = score
        best_seq = seq
        best_step = step
        print(f"  ⭐ New best!")

    gc.collect(); torch.cuda.empty_cache()

print()
print("=" * 60)
print(f"Best result: Step {best_step}  |  Score: {best_score:.3f}")
print(f"Best sequence: {best_seq}")
print("=" * 60)

Starting sequence: YMGFVKRKKDEMNHWIAASCASADARHVLTLHDLFWDRNVQIAHFFSFSF...
Hysteresis mode:   templates

Step 0: Initial fold (no templates)
  pLDDT: 49.3
  Score: 0.056

Step 1: Design + Fold
  [MPNN] Designing new sequence...
  New seq: EKKEIEEKLKEIEARCEKYKDTEDPSTVISLDKMLNDKEVQLRLLTSDSP...
  [Fold] Predicting structure (hysteresis=templates)...
  pLDDT: 83.1  |  Score: 0.160
  ⭐ New best!

Step 2: Design + Fold
  [MPNN] Designing new sequence...
  New seq: ERAAIEEELARLEARCAAYRDKRFPSTEISRERMLTDPAVQRRLLTSESP...
  [Fold] Predicting structure (hysteresis=templates)...
  pLDDT: 90.6  |  Score: 0.179
  ⭐ New best!

Step 3: Design + Fold
  [MPNN] Designing new sequence...
  New seq: EDAAIEARLAELRARCAAYRGREFPAEAISLERMLTDPAVQRRLLTSTSP...
  [Fold] Predicting structure (hysteresis=templates)...
  pLDDT: 90.4  |  Score: 0.177

Best result: Step 2  |  Score: 0.179
Best sequence: ERAAIEEELARLEARCAAYRDKRFPSTEISRERMLTDPAVQRRLLTSESPGDLANLAALLEYLRKEGRLLPYLAEVLSGA


In [ ]:
# @title Visualize final structure
folder.plot(color_by="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

### Example 5b: Using the Built-in `optimize_protein_design()` Function

The manual loop above is great for learning, but for production work, use the built-in function which handles edge cases, memory management, and final validation.

In [ ]:
# @title Example 5b: Production design with optimize_protein_design()
# @markdown ### Design Settings
jobname = "tutorial_design"    # @param {type:"string"}
length = 100                   # @param {type:"integer"}
n_cycles = 3                   # @param {type:"integer"}
hysteresis_mode = "templates"  # @param ["templates", "esm", "partial_diffusion", "none"]
temperature = 0.1              # @param {type:"number"}
repredict = True               # @param {type:"boolean"}

# Parse hysteresis mode into function arguments
opts = dict(
    use_esm=False,
    use_esm_target=False,
    pde_cutoff_intra=0.0,
    pde_cutoff_inter=0.0,
    partial_diffusion=0.0,
)
if hysteresis_mode == "templates":
    opts["pde_cutoff_intra"] = 1.5
    opts["pde_cutoff_inter"] = 3.0
elif hysteresis_mode == "esm":
    opts["use_esm"] = True
elif hysteresis_mode == "partial_diffusion":
    opts["partial_diffusion"] = 0.5

initial_seq = sample_seq(length, frac_X=0.0)

result = optimize_protein_design(
    folder=folder,
    designer=designer,
    initial_seq=initial_seq,
    target_seq=None,         # None = unconditional design
    prefix=jobname,
    n_steps=n_cycles,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    temperature=temperature,
    randomize_template_sequence=True,
    final_validation=repredict,
    **opts
)

folder.full_cleanup()
gc.collect(); torch.cuda.empty_cache()

if result:
    print()
    print(f"Best sequence: {result['seq']}")
    folder.restore_state(result["state"])

tutorial_design | Step 0: score=0.079 plddt=0.5 ptm=0.397 pae=13.92
tutorial_design | Step 1: score=0.163 plddt=0.8 ptm=0.816 pae=4.09 rmsd=0.81
tutorial_design | Step 2: score=0.179 plddt=0.9 ptm=0.896 pae=2.63 rmsd=0.35
tutorial_design | Step 3: score=0.182 plddt=0.9 ptm=0.911 pae=2.49 rmsd=0.30
tutorial_design | Validation: score=0.181 plddt=0.9 ptm=0.903 pae=2.71 rmsd=0.22

Best sequence: MKEERVVGVEEVRKYLEENFPKELADLLLNPGSEEELEKKIKEINEKGEKNGEGKILSWSLSEEDGVKTLRLEIEYKGKKYTITVRHRDGELELTVKYEE


In [ ]:
# @title Visualize optimized design
if result:
    folder.plot(color_by="plddt")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Quick Reference

### ChaiFolder Options Dictionary

Every chain passed to `prep_inputs()` has an options dict:

```python
options = {
    "use_esm": False,                      # ESM2 embeddings
    "replace_x_with_mask": False,          # treat X as <mask> for ESM
    "template_pdb": None,                  # path to template CIF/PDB
    "template_chain_id": None,             # chain ID in template
    "randomize_template_sequence": False,  # randomize template residue types
    "align": 1.0,                          # alignment weight during diffusion
    "cyclic": False,                       # cyclic backbone
}
```

### Entity Types
- `"protein"` — amino acid chain
- `"ligand"` — small molecule (SMILES string)
- `"rna"` / `"dna"` — nucleic acid chains

### Hysteresis Strategy Cheat Sheet

| Task | Hysteresis | ESM | Temperature |
|---|---|---|---|
| Novel fold (unconditional) | `templates` | Off | 0.1 |
| Protein binder | `templates` | Off / On for target | 0.1 |
| Ligand binder | `esm` | On | 0.01 |
| Refinement | `partial_diffusion` | Optional | 0.1 |

### Confidence Metrics

| Metric | Range | Good values | Meaning |
|---|---|---|---|
| pLDDT | 0–100 | >70 | Per-residue confidence |
| pTM | 0–1 | >0.5 | Overall fold confidence |
| ipTM | 0–1 | >0.5 | Interface confidence (multi-chain) |
| PAE | 0–32 Å | <5 | Predicted alignment error |
| Ranking score | 0–1 | higher = better | Composite score for selection |

---

## Download Results

In [ ]:
# @title Download all CIF files as zip
from google.colab import files
import os

cif_files = [f for f in os.listdir(".") if f.endswith(".cif")]
dirs_with_cifs = [d for d in os.listdir(".") if os.path.isdir(d)]
for d in dirs_with_cifs:
    for f in os.listdir(d):
        if f.endswith(".cif"):
            cif_files.append(os.path.join(d, f))

if cif_files:
    os.system("zip -r tutorial_results.zip *.cif *//*.cif 2>/dev/null")
    files.download("tutorial_results.zip")
    print(f"Downloaded {len(cif_files)} structure files")
else:
    print("No CIF files found. Run the examples first!")